# Berramdane Model V9.4
## Quantum Double-Slit with Angular Analysis
**Author:** Al Moalim Berramdane
**Description:** Adds precision angular measurements (Interference & Diffraction) to the core V9.3 model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox, IntSlider
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

h, m, L_total = 6.626e-34, 9.109e-31, 2.2

def de_broglie_wavelength(v): 
    return h / (m * v)

def compute_angles(v_mean, a_width, d_slit):
    lam = de_broglie_wavelength(v_mean)
    # زاوية أول هدب تداخل: θ = λ / d
    theta_i_rad = lam / d_slit
    theta_i_deg = theta_i_rad * 180 / np.pi
    # زاوية أول عقدة حيود: θ = λ / a
    theta_d_rad = lam / a_width
    theta_d_deg = theta_d_rad * 180 / np.pi
    return theta_i_rad, theta_i_deg, theta_d_rad, theta_d_deg

def double_slit_intensity_single_velocity(x, v_par, L, a_width, d_slit):
    lam = de_broglie_wavelength(v_par)
    beta = (np.pi * d_slit * x) / (lam * L)
    alpha = (np.pi * a_width * x) / (lam * L)
    return np.cos(beta)**2 * np.sinc(alpha / np.pi)**2

def particle_like_pattern(x, v_par, L, a_width, d_slit):
    lam = de_broglie_wavelength(v_par)
    sigma = a_width * L / lam
    return 0.5 * (np.exp(-(x + d_slit/2)**2 / (2 * sigma**2)) + np.exp(-(x - d_slit/2)**2 / (2 * sigma**2)))

@interact(
    v_mean=FloatSlider(value=5.8e5, min=2e5, max=1.2e6, step=0.1e5, description='Velocity (m/s)'),
    a_width=FloatSlider(value=0.72e-6, min=0.2e-6, max=1.5e-6, step=0.01e-6, description='Slit Width (m)'),
    d_slit=FloatSlider(value=2.45e-6, min=1.0e-6, max=5.0e-6, step=0.05e-6, description='Separation (m)'),
    observer_active=Checkbox(value=False, description='Detector ON'),
    meas_strength=FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description='Strength')
)
def interactive_lab(v_mean, a_width, d_slit, observer_active, meas_strength):
    if a_width >= d_slit: a_width = d_slit * 0.99
    
    x = np.linspace(-0.005, 0.005, 1000)
    lam = de_broglie_wavelength(v_mean)
    
    # حساب الزوايا (إضافة المعلم بالرمضان)
    ti_rad, ti_deg, td_rad, td_deg = compute_angles(v_mean, a_width, d_slit)
    
    I_interf = double_slit_intensity_single_velocity(x, v_mean, L_total, a_width, d_slit)
    I_part = particle_like_pattern(x, v_mean, L_total, a_width, d_slit)
    
    I = ((1 - meas_strength) * I_interf + meas_strength * I_part) if observer_active else I_interf
    I /= np.max(I)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # الرسم البياني
    ax1.plot(x*1000, I, 'b-', lw=1.5)
    ax1.set_title(f'Berramdane Model V9.4')
    ax1.set_xlabel('Position (mm)')
    ax1.grid(True, alpha=0.3)
    
    # لوحة المعلومات التقنية والزوايا
    info = (f"λ = {lam*1e9:.3f} nm\n\n"
            f"📐 Interference Max Angle:\n   {ti_deg:.4f}° ({ti_rad:.4e} rad)\n\n"
            f"📐 Diffraction Min Angle:\n   {td_deg:.4f}° ({td_rad:.4e} rad)\n\n"
            f"Fringe Spacing: {(lam * L_total / d_slit)*1000:.3f} mm")
    
    ax2.text(0.05, 0.5, info, transform=ax2.transAxes, fontsize=12, 
             verticalalignment='center', family='monospace', 
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    ax2.axis('off')
    
    plt.show()
    print(f"✅ Calculation complete for v = {v_mean/1e3:.0f} km/s")